# Refrigerated Warehouse Cooling Energy Estimation
**INSTRUCTOR REFERENCE | AI Services MS2 — Supplementary Regression Practice | 60 marks | 20 tests**

A cold-storage company records hourly refrigeration activity. Build a reusable workflow that estimates cooling energy use (kWh). The dataset has deliberate data-quality issues.

**Workflow:** clean data → engineer features → create a fitted numeric/categorical pipeline → compare two regressors → save the selected model → predict future records.

Run the definition cells before using the training and prediction cells. Keep the original CSV files unchanged.

## 1. Imports and feature groups

In [ ]:
import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "energy_consumed_kwh"

RAW_NUMERIC = [
    "ambient_temperature_c", "chamber_setpoint_c", "goods_load_tonnes",
    "operating_hours", "door_open_minutes", "compressor_load_pct"
]
CATEGORICAL = ["site_code", "refrigerant_type", "service_state", "shift_code"]
NUMERIC_FEATURES = RAW_NUMERIC + [
    "start_hour", "is_weekend", "temperature_gap_c", "load_rate_tph"
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL


## 2. Seven independently testable functions and two workflows

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    """Clean records and engineer the same features for every CSV; return a DataFrame."""
    df = pd.read_csv(filename, dtype={"reading_id": "string", "shift_code": "string"})
    df = df.drop_duplicates("reading_id", keep="first").copy()

    timestamps = pd.to_datetime(
        df["reading_start_ts"], format="mixed", dayfirst=True, errors="coerce"
    )

    df = df.loc[timestamps.notna()].copy()
    df["reading_start_ts"] = timestamps.loc[df.index]

    for col in RAW_NUMERIC:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    df[RAW_NUMERIC] = df[RAW_NUMERIC].replace([np.inf, -np.inf], np.nan)

    for col in CATEGORICAL:
        value = df[col].astype("string").str.strip().str.lower()
        value = value.str.replace(r"^([+-]?\d+)\.0+$", r"\1", regex=True)

        df[col] = value.replace("", pd.NA).fillna("unknown").astype(str)

    df["start_hour"] = df["reading_start_ts"].dt.hour

    df["is_weekend"] = (df["reading_start_ts"].dt.dayofweek >= 5).astype(int)
    df["temperature_gap_c"] = ( df["ambient_temperature_c"] - df["chamber_setpoint_c"] )


    safe_hours = df["operating_hours"].where(df["operating_hours"] > 0)
    df["load_rate_tph"] = (df["goods_load_tonnes"] / safe_hours).replace([np.inf, -np.inf], np.nan)


    return df.reset_index(drop=True)


def build_pipeline(algorithm: BaseEstimator) -> Pipeline:
    """Return an unfitted full model pipeline with numeric and categorical branches."""
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)
    ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", algorithm)
    ])


def prepare_datasets(
    train_file: str, test_file: str
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Return X_train, X_test, y_train, y_test with invalid targets excluded."""
    train_df = load_data(train_file)
    test_df = load_data(test_file)
    for df in (train_df, test_df):
        df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
        df[TARGET] = df[TARGET].replace([np.inf, -np.inf], np.nan)
        df.dropna(subset=[TARGET], inplace=True)
        df.drop(df[df[TARGET] < 0].index, inplace=True)
    return (
        train_df[FEATURES], test_df[FEATURES],
        train_df[TARGET], test_df[TARGET]
    )


def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    """Return separately fitted Ridge and Random Forest pipelines."""
    models = {
        "ridge_regression": Ridge(alpha=1.0),
        "random_forest": RandomForestRegressor(
            n_estimators=120, max_depth=12, min_samples_leaf=2,
            random_state=42, n_jobs=-1
        )
    }
    trained_models = {}
    for name, algorithm in models.items():
        pipeline = build_pipeline(algorithm)
        pipeline.fit(X_train, y_train)
        trained_models[name] = pipeline
    return trained_models


def evaluate_models(
    trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series
) -> pd.DataFrame:
    """Return MAE/RMSE/R2 comparison ranked by increasing MAE."""
    results = []
    for name, pipeline in trained_models.items():
        predictions = pipeline.predict(X_test)
        results.append({
            "model_name": name,
            "mae": mean_absolute_error(y_test, predictions),
            "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
            "r2": r2_score(y_test, predictions)
        })
    comparison = pd.DataFrame(results)
    return comparison.sort_values("mae").reset_index(drop=True)


def select_best_model(
    comparison: pd.DataFrame, trained_models: dict[str, Pipeline]
) -> tuple[str, Pipeline]:
    """Return the first ranked model's name and its already fitted pipeline."""
    best_name = comparison.iloc[0]["model_name"]
    return best_name, trained_models[best_name]


def predict_energy(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    """Return nonnegative one-decimal energy predictions for cleaned queued records."""
    predictions = np.maximum(model.predict(df[FEATURES]), 0).round(1)
    return pd.DataFrame({
        "reading_id": df["reading_id"].to_numpy(),
        "predicted_energy_kwh": predictions
    })


def train() -> tuple[str, pd.DataFrame]:
    """Train both models, save the best pipeline and comparison CSV, and return summary."""
    X_train, X_test, y_train, y_test = prepare_datasets(
        "cooling_train.csv", "cooling_test.csv"
    )
    trained_models = train_models(X_train, y_train)
    comparison = evaluate_models(trained_models, X_test, y_test)
    best_name, best_model = select_best_model(comparison, trained_models)
    joblib.dump(best_model, "best_model.pkl")
    comparison.to_csv("model_comparison.csv", index=False)
    print(comparison.round(2).to_string(index=False))
    print("Selected model:", best_name)
    return best_name, comparison


def predict() -> pd.DataFrame:
    """Load saved pipeline and write independent batch predictions without refitting."""
    queued_df = load_data("cooling_predict.csv")
    fitted_model = joblib.load("best_model.pkl")
    result = predict_energy(fitted_model, queued_df)
    result.to_csv("cooling_predictions.csv", index=False)
    print(result.head().to_string(index=False))
    return result


## 3. Train the two candidate models (creates saved outputs)

In [ ]:
# best_name, comparison = train()

## 4. Independently predict queued runs (does not retrain)

In [ ]:
# predictions = predict()